# Test the feature-extraction workflow

Exercises every public function in `feature_extraction_workflow.extract_features` against a sample of `df_items`. Each section verifies one stage of the pipeline before running the end-to-end driver.

In [91]:
import sys
from pathlib import Path

# Make the package importable when running from this folder
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
from nltk.stem import WordNetLemmatizer

from feature_extraction_workflow import (
    build_global_filter_regex,
    build_stopwords,
    clean_text,
    ensure_cat_columns,
    extract_features,
    filter_by_cat_3,
    merge_extracted,
    parse_list_string,
    remove_global_filters,
    run_feature_extraction,
)

pd.options.display.max_colwidth = 80
pd.options.display.max_columns = None

## 1. Load data

Read a sample of items plus the schema and global-filter configs. A 50k-row sample keeps the notebook fast while still hitting most categories.

In [92]:
DATA_DIR = PROJECT_ROOT / 'data'

df_items = pd.read_csv(
    DATA_DIR / 'meta_Home_and_Kitchen_filtered.csv',
    low_memory=False,
    # nrows=50_000,
).drop_duplicates()

print(f'df_items: {len(df_items):,} rows')
df_items[['asin', 'title', 'category', 'description', 'feature']].head(3)

df_items: 1,285,392 rows


,asin,title,category,description,feature
0,0001487795,You Are Special Today Red Plate [With Red Pen],"['Home & Kitchen', 'Kitchen & Dining', 'Dining & Entertaining', 'Dinnerware'...",['It was a time honored tradition among the early American families that whe...,[]
1,0002020300,Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy,"['Home & Kitchen', 'Home Dcor', 'Candles & Holders', 'Candles']",['VICKS INHALER relieves stuffy noses helps sinus congestion breathe easy gr...,[]
2,0006564224,Artistic Churchware Communion Cup Filler: RW525,"['Home & Kitchen', 'Kitchen & Dining', 'Dining & Entertaining', 'Glassware &...","['16 oz squeeze bottle, 1 lb.']","['Religious Supply Center', 'RW-525', 'Communion Cup Filler']"


In [93]:
with open(DATA_DIR / 'master_metadata.json') as f:
    master_metadata = json.load(f)

with open(DATA_DIR / 'global_filters.json') as f:
    global_filters = json.load(f)

print(f'master_metadata categories: {len(master_metadata)}')
print(f'global_filters phrases: {len(global_filters)}')
print(f'\nExample category schema (cat_3 = {next(iter(master_metadata))!r}):')
first_cat = next(iter(master_metadata))
print(json.dumps({first_cat: master_metadata[first_cat]}, indent=2)[:500])

master_metadata categories: 69
global_filters phrases: 41

Example category schema (cat_3 = 'Bakeware'):
{
  "Bakeware": {
    "Brand": {
      "type": "dictionary",
      "values": [
        "ann clark",
        "cybrtrayd",
        "wilton",
        "fat daddio",
        "nordic ware",
        "le creuset",
        "bia cordon bleu",
        "fox run",
        "paderno world cuisine",
        "ck product",
        "villeroy & boch",
        "coppergifts",
        "first impression",
        "ateco",
        "meri meri",
        "matfer bourgeat",
        "pampered chef",
        "chicago metallic


## 2. Test individual helpers

Smoke-test each building block in isolation before chaining them.

In [94]:
# parse_list_string: stringified list -> single space-joined string
samples = [
    "['Religious Supply Center', 'RW-525', 'Communion Cup Filler']",
    "['Set of 15 Blue, Red & Black Ball Pen']",
    '[]',
    None,
    'plain string passthrough',
]
for s in samples:
    print(f'INPUT : {s!r}')
    print(f'OUTPUT: {parse_list_string(s)!r}\n')

INPUT : "['Religious Supply Center', 'RW-525', 'Communion Cup Filler']"
OUTPUT: 'Religious Supply Center RW-525 Communion Cup Filler'

INPUT : "['Set of 15 Blue, Red & Black Ball Pen']"
OUTPUT: 'Set of 15 Blue, Red & Black Ball Pen'

INPUT : '[]'
OUTPUT: ''

INPUT : None
OUTPUT: ''

INPUT : 'plain string passthrough'
OUTPUT: 'plain string passthrough'



In [95]:
# clean_text: lowercase, number-words to digits, strip noise, lemmatize, drop stopwords
stop_words = build_stopwords()
lemmatizer = WordNetLemmatizer()

samples = [
    'You Are Special Today Red Plate [With Red Pen]',
    'Five-Drawer Wooden Storage Cabinet, 12-pc Set',
    'Nikola Tesla Photo&hellip;Quotes Poster Print (12 inch X 18 inch, Rolled)',
    'Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy',
    None,
    '',
]
for s in samples:
    print(f'BEFORE: {s!r}')
    print(f'AFTER : {clean_text(s, stop_words, lemmatizer)!r}\n')

BEFORE: 'You Are Special Today Red Plate [With Red Pen]'
AFTER : 'special today red plate red pen'

BEFORE: 'Five-Drawer Wooden Storage Cabinet, 12-pc Set'
AFTER : '5-drawer wooden storage cabinet 12 pc set'

BEFORE: 'Nikola Tesla Photo&hellip;Quotes Poster Print (12 inch X 18 inch, Rolled)'
AFTER : 'nikola tesla photo hellip quote poster print 12 inch x 18 inch rolled'

BEFORE: 'Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy'
AFTER : 'vicks inhaler relief cold sinus nasal congestion allergy'

BEFORE: None
AFTER : ''

BEFORE: ''
AFTER : ''



In [96]:
# build_global_filter_regex + remove_global_filters
filter_regex = build_global_filter_regex(global_filters)

samples = [
    'great workplace poster 100% satisfaction guaranteed',
    'beautiful candle perfect gift for any occasion',
    'no filtering needed here',
]
for s in samples:
    print(f'BEFORE: {s!r}')
    print(f'AFTER : {remove_global_filters(s, filter_regex)!r}\n')

BEFORE: 'great workplace poster 100% satisfaction guaranteed'
AFTER : 'great workplace poster 100% satisfaction guaranteed'

BEFORE: 'beautiful candle perfect gift for any occasion'
AFTER : 'beautiful candle perfect  for any occasion'

BEFORE: 'no filtering needed here'
AFTER : 'no filtering needed here'



In [97]:
# ensure_cat_columns: parse stringified category list into cat_1..cat_6
df_with_cats = ensure_cat_columns(df_items)
cat_cols = [f'cat_{i+1}' for i in range(6)]

print('cat_1..cat_6 created:', all(c in df_with_cats.columns for c in cat_cols))
print(f'\ncat_3 non-null: {df_with_cats["cat_3"].notna().sum():,} / {len(df_with_cats):,}')
df_with_cats[['asin'] + cat_cols].head(5)

cat_1..cat_6 created: True

cat_3 non-null: 1,243,641 / 1,285,392


,asin,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6
0,0001487795,Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates
1,0002020300,Home & Kitchen,Home Dcor,Candles & Holders,Candles,None,None
2,0006564224,Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Wine & Champagne Glasses,None
3,0009046461,Home & Kitchen,Bath,Bathroom Accessories,None,None,None
4,0234937912,Home & Kitchen,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense,None


In [98]:
# filter_by_cat_3: keep only rows whose cat_3 has an extraction schema
df_filtered = filter_by_cat_3(df_with_cats, master_metadata)

print(f'Before filter: {len(df_with_cats):,}')
print(f'After filter : {len(df_filtered):,} ({len(df_filtered)/len(df_with_cats)*100:.1f}% kept)')
print(f'\nUnique cat_3 values represented: {df_filtered["cat_3"].nunique()} / {len(master_metadata)}')
print(f'\nTop cat_3 values:')
print(df_filtered['cat_3'].value_counts().head(10))

Before filter: 1,285,392
After filter : 1,134,566 (88.3% kept)

Unique cat_3 values represented: 69 / 69

Top cat_3 values:
cat_3
Home Dcor Accents                       158450
Dining & Entertaining                   142544
Posters & Prints                        102456
Kitchen Utensils & Gadgets               77208
Storage & Organization                   42317
Kitchen & Table Linens                   39682
Bakeware                                 38454
Decorative Pillows, Inserts & Covers     37478
Bathroom Accessories                     36675
Candles & Holders                        34266
Name: count, dtype: int64


In [99]:
# extract_features on a few cleaned titles
sample = df_filtered[['asin', 'title', 'cat_3']].dropna(subset=['title', 'cat_3']).head(5).copy()
sample['title_cleaned'] = sample['title'].apply(
    lambda t: remove_global_filters(
        clean_text(t, stop_words, lemmatizer),
        filter_regex,
    )
)
sample['features'] = sample.apply(
    lambda r: extract_features(r['title_cleaned'], r['cat_3'], master_metadata),
    axis=1,
)

for _, row in sample.iterrows():
    print(f"cat_3   : {row['cat_3']}")
    print(f"title   : {row['title'][:80]}")
    print(f"cleaned : {row['title_cleaned'][:80]}")
    print(f"features: {row['features']}")
    print()

cat_3   : Dining & Entertaining
title   : You Are Special Today Red Plate [With Red Pen]
cleaned : special today red plate red pen
features: {'Color': 'red'}

cat_3   : Candles & Holders
title   : Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy
cleaned : vicks inhaler relief cold sinus nasal congestion allergy
features: {}

cat_3   : Dining & Entertaining
title   : Artistic Churchware Communion Cup Filler: RW525
cleaned : artistic churchware communion cup filler rw525
features: {'Product_Type': 'cup'}

cat_3   : Bathroom Accessories
title   : 4 BARS! Mysore Sandal Soap 70grams FAST SHIPPING
cleaned : 4 bar mysore sandal soap 70grams fast shipping
features: {}

cat_3   : Home Fragrance
title   : AROGYA VATI (40gm) by popeye seller
cleaned : arogya vati 40gm popeye seller
features: {'Capacity_Volume': '40gm'}



## 3. End-to-end driver

`run_feature_extraction` chains every stage. Pass paths instead of pre-loaded objects to verify the JSON-loading branch works too.

In [100]:
# Edit this list to add or remove text columns to extract from.
# `list_columns` is the subset whose raw values are stringified lists.
text_columns = ['title', 'description', 'feature']
list_columns = ['description', 'feature']

df_result = run_feature_extraction(
    df_items,
    text_columns=text_columns,
    master_metadata=str(DATA_DIR / 'master_metadata.json'),
    global_filters=str(DATA_DIR / 'global_filters.json'),
    list_columns=list_columns,
    priority=text_columns,   # highest-to-lowest; defaults to text_columns order
)

print(f'Result shape: {df_result.shape}')
print(f'\nNew columns:')
expected = (
    [f'cat_{i+1}' for i in range(6)]
    + [f'{c}_cleaned' for c in text_columns]
    + [f'extracted_features_{c}' for c in text_columns]
    + ['extracted_features']
)
for col in expected:
    print(f'  {col:35s} present: {col in df_result.columns}')

Result shape: (1134566, 25)

New columns:
  cat_1                               present: True
  cat_2                               present: True
  cat_3                               present: True
  cat_4                               present: True
  cat_5                               present: True
  cat_6                               present: True
  title_cleaned                       present: True
  description_cleaned                 present: True
  feature_cleaned                     present: True
  extracted_features_title            present: True
  extracted_features_description      present: True
  extracted_features_feature          present: True
  extracted_features                  present: True


In [101]:
# Per-source vs combined coverage
from collections import Counter

total = len(df_result)
for src in ['title', 'description', 'feature']:
    n = (df_result[f'extracted_features_{src}'].apply(len) > 0).sum()
    print(f'  {src:12s}: {n:,} / {total:,} ({n/total*100:.1f}%)')

n_combined = (df_result['extracted_features'].apply(len) > 0).sum()
print(f'  combined    : {n_combined:,} / {total:,} ({n_combined/total*100:.1f}%)')

field_counts = Counter()
for d in df_result['extracted_features']:
    field_counts.update(d.keys())
print('\nTop fields extracted (combined):')
for field, count in field_counts.most_common(10):
    print(f'  {field:20s} {count:,} ({count/total*100:.1f}%)')

  title       : 1,048,500 / 1,134,566 (92.4%)
  description : 915,857 / 1,134,566 (80.7%)
  feature     : 876,231 / 1,134,566 (77.2%)
  combined    : 1,112,733 / 1,134,566 (98.1%)

Top fields extracted (combined):
  Product_Type         838,262 (73.9%)
  Material             761,739 (67.1%)
  Features             638,614 (56.3%)
  Dimensions           591,890 (52.2%)
  Color                530,245 (46.7%)
  Piece_Count          251,125 (22.1%)
  Theme                181,559 (16.0%)
  Brand                152,854 (13.5%)
  Capacity_Volume      124,755 (11.0%)
  Size                 81,941 (7.2%)


In [102]:
# Inspect a few fully-processed rows
view_cols = ['asin', 'cat_3', 'title_cleaned', 'extracted_features']
df_result[df_result['extracted_features'].apply(len) > 0][view_cols].head(5)

,asin,cat_3,title_cleaned,extracted_features
0,0001487795,Dining & Entertaining,special today red plate red pen,{'Color': 'red'}
2,0006564224,Dining & Entertaining,artistic churchware communion cup filler rw525,"{'Product_Type': 'cup', 'Capacity_Volume': '16 oz'}"
3,0009046461,Bathroom Accessories,4 bar mysore sandal soap 70grams fast shipping,{'Features': 'natural'}
4,0234937912,Home Fragrance,arogya vati 40gm popeye seller,"{'Features': 'natural', 'Capacity_Volume': '40gm'}"
5,0250459655,Posters & Prints,nikola tesla photo nikola tesla quote poster print 12 inch x 18 inch rolled,"{'Features': 'framed', 'Dimensions': '12 inch', 'Product_Type': 'poster prin..."


### 3.1 Single-pass extraction: concatenate then extract  *(commented out — skip by default)*

**Purpose.** This section is a side-by-side experiment comparing two extraction strategies:

1. **Per-source then merge** (the default in section 3): clean each text column individually, run `extract_features` on each, merge the three dicts with priority `title > description > feature`.
2. **Concat then extract once** (this section): concatenate the cleaned text from all sources into one string per row, then run `extract_features` a single time.

It then reports per-field coverage, exact agreement, and disagreement examples between the two outputs.

**Why it's commented out.** It runs `extract_features` over every row a second time, which is the slowest step in the pipeline. The downstream pipeline (sections 4–6) only consumes the per-source-merge result — the single-pass output is not used. Re-enable the cells below if you want to run the comparison; uncomment by removing the leading `# ` from each line.

In [103]:
# import json
# from feature_extraction_workflow import extract_features
#
# with open(DATA_DIR / 'master_metadata.json') as f:
#     master_metadata = json.load(f)
#
# df_merged = df_result.copy()
#
# # Concatenate the already-cleaned text from all sources into one string per row
# df_merged['merged_text'] = (
#     df_merged[[f'{c}_cleaned' for c in text_columns]]
#     .fillna('')
#     .agg(' '.join, axis=1)
#     .str.replace(r'\s+', ' ', regex=True)
#     .str.strip()
# )
#
# # Run extract_features once on the concatenated text
# df_merged['extracted_features_merged'] = df_merged.apply(
#     lambda r: extract_features(r['merged_text'], r.get('cat_3'), master_metadata),
#     axis=1,
# )
#
# print(f'Built merged_text and extracted_features_merged columns')
# print(f'Sample merged_text (first row): {df_merged["merged_text"].iloc[0][:120]}')
# print(f'Sample features_merged:         {df_merged["extracted_features_merged"].iloc[0]}')

In [104]:
# from collections import Counter
#
# total = len(df_merged)
# n_split  = (df_merged['extracted_features'].apply(len) > 0).sum()
# n_merged = (df_merged['extracted_features_merged'].apply(len) > 0).sum()
#
# print(f'Per-source then merge (section 3): {n_split:,} / {total:,} ({n_split/total*100:.1f}%)')
# print(f'Concat then extract once (3.1):    {n_merged:,} / {total:,} ({n_merged/total*100:.1f}%)')
#
# fc_split = Counter()
# fc_merged = Counter()
# for d in df_merged['extracted_features']:
#     fc_split.update(d.keys())
# for d in df_merged['extracted_features_merged']:
#     fc_merged.update(d.keys())
#
# print(f'\nPer-field counts (split | single-pass | diff):')
# all_fields = sorted(set(fc_split) | set(fc_merged), key=lambda f: -fc_merged.get(f, 0))
# for f in all_fields:
#     s = fc_split[f]
#     m = fc_merged[f]
#     print(f'  {f:22s} {s:>7,} | {m:>7,} | {m - s:+,}')

In [105]:
# # How often do the two approaches agree?
# both_empty = identical = differ_full = only_split = only_merged = 0
# for d_s, d_m in zip(df_merged['extracted_features'], df_merged['extracted_features_merged']):
#     if not d_s and not d_m:
#         both_empty += 1
#     elif d_s == d_m:
#         identical += 1
#     elif d_s and not d_m:
#         only_split += 1
#     elif d_m and not d_s:
#         only_merged += 1
#     else:
#         differ_full += 1
#
# total = len(df_merged)
# print(f'Both empty (no extraction):              {both_empty:,} ({both_empty/total*100:.1f}%)')
# print(f'Identical dicts:                         {identical:,} ({identical/total*100:.1f}%)')
# print(f'Only per-source has features:            {only_split:,}')
# print(f'Only single-pass has features:           {only_merged:,}')
# print(f'Both populated but disagree:             {differ_full:,}')

In [106]:
# # Show a handful of disagreement examples to eyeball whether one is clearly better
# def _is_disagreement(r):
#     s = r['extracted_features']
#     m = r['extracted_features_merged']
#     if not s and not m:
#         return False
#     return s != m
#
# mask = df_merged.apply(_is_disagreement, axis=1)
# sample = df_merged[mask].head(50)
# for _, row in sample.iterrows():
#     print(f"\nasin: {row['asin']} | cat_3: {row['cat_3']}")
#     print(f"  title  : {row['title_cleaned'][:90]}")
#     print(f"  per-src: {row['extracted_features']}")
#     print(f"  merged : {row['extracted_features_merged']}")

## 4. Expand to per-field columns

Turn the merged `extracted_features` dict into typed columns: one column per field (`Color`, `Material`, `Dimensions`, …), `<field>_numeric` + `<field>_unit` for capacity/weight/etc., `dimension_1/2/3/_unit` for dimensions, and standardized units via `UNIT_MAP`. This is what produces the table you saw in `create_features.ipynb`.

In [107]:
from feature_extraction_workflow import expand_features

df_expanded = expand_features(df_result)

print(f'Expanded shape: {df_expanded.shape}')

# Show numeric/unit columns produced
numeric_cols = [c for c in df_expanded.columns if c.endswith('_numeric')]
unit_cols = [c for c in df_expanded.columns if c.endswith('_unit')]
dim_cols = [c for c in df_expanded.columns if c.startswith('dimension_')]

print(f'\nNumeric columns ({len(numeric_cols)}):')
for c in numeric_cols:
    n = df_expanded[c].notna().sum()
    print(f'  {c:30s} {n:,} non-null')

print(f'\nUnit columns ({len(unit_cols)}):')
for c in unit_cols:
    units = df_expanded[c].dropna().unique()
    print(f'  {c:30s} units: {sorted(units)[:10]}')

print(f'\nDimension columns:')
for c in dim_cols:
    n = df_expanded[c].notna().sum()
    print(f'  {c:20s} {n:,} non-null')

Expanded shape: (1134566, 72)

Numeric columns (9):
  capacity_numeric               4,026 non-null
  capacity_volume_numeric        124,755 non-null
  piece_count_numeric            157,366 non-null
  thread_count_numeric           19,346 non-null
  weight_numeric                 252 non-null
  bar_pressure_numeric           460 non-null
  capacity_cups_numeric          2,701 non-null
  stage_count_numeric            362 non-null
  voltage_numeric                2,363 non-null

Unit columns (6):
  capacity_unit                  units: ['cup', 'l', 'oz']
  capacity_volume_unit           units: ['bottle', 'cubic foot', 'cup', 'fl oz', 'g', 'gal', 'l', 'lb', 'ml', 'oz']
  piece_count_unit               units: ['bottle', 'capacity', 'chair', 'cone', 'count', 'door', 'drawer', 'dz', 'hook', 'in 1']
  thread_count_unit              units: ['count', 'series', 'thread count']
  weight_unit                    units: ['g', 'lb']
  dimension_unit                 units: ['cm', 'cm - m', 'cm --m',

In [108]:
# Spot-check parsing: rows where Capacity_Volume / Dimensions were populated
view_cols = ['asin', 'cat_3', 'Capacity_Volume', 'capacity_volume_numeric', 'capacity_volume_unit']
view_cols = [c for c in view_cols if c in df_expanded.columns]
df_expanded[df_expanded.get('capacity_volume_numeric', pd.Series(dtype=float)).notna()][view_cols].head(5)

,asin,cat_3,Capacity_Volume,capacity_volume_numeric,capacity_volume_unit
2,0006564224,Dining & Entertaining,16 oz,16.0,oz
4,0234937912,Home Fragrance,40gm,40.0,g
17,0641965974,Dining & Entertaining,12 oz,12.0,oz
40,0980092191,Small Appliances,1 cup,1.0,cup
76,159617143X,Dining & Entertaining,5oz,5.0,oz


In [109]:
view_cols = ['asin', 'cat_3', 'Dimensions', 'dimension_1', 'dimension_2', 'dimension_3', 'dimension_unit']
view_cols = [c for c in view_cols if c in df_expanded.columns]
df_expanded[df_expanded.get('dimension_1', pd.Series(dtype=float)).notna()][view_cols].head(5)

,asin,cat_3,Dimensions,dimension_1,dimension_2,dimension_3,dimension_unit
5,0250459655,Posters & Prints,12 inch,12.0,NaN,NaN,in
11,0560467893,Home Dcor Accents,20 inch,20.0,NaN,NaN,in
12,0587228237,Posters & Prints,12x18,18.0,12.0,NaN,NaN
13,0594496780,Blankets & Throws,50 x 60,60.0,50.0,NaN,NaN
15,0635118777,Kitchen Utensils & Gadgets,2 x 5,5.0,2.0,NaN,NaN


## 5. Clean numeric features to valid ranges

`clean_numeric_ranges` keeps the original columns intact and adds `<col>_cleaned` versions where out-of-range values are set to NaN. Same range table as `notebooks/analyze_features.ipynb`.

In [110]:
from feature_extraction_workflow import clean_numeric_ranges, VALID_RANGES

df_clean = clean_numeric_ranges(df_expanded)

print('Range filter applied — rows kept inside [low, high], else NaN:\n')
for col, (low, high) in VALID_RANGES.items():
    if col not in df_clean.columns:
        print(f'  {col}: not present in df, skipped')
        continue
    before = df_clean[col].notna().sum()
    after = df_clean[f'{col}_cleaned'].notna().sum()
    removed = before - after
    pct = (removed / before * 100) if before else 0.0
    print(f'  {col} [{low}-{high}]: {before:,} -> {after:,} ({removed:,} dropped, {pct:.1f}% of non-null)')

Range filter applied — rows kept inside [low, high], else NaN:

  bar_pressure_numeric [9-19]: 460 -> 410 (50 dropped, 10.9% of non-null)
  capacity_cups_numeric [1-14]: 2,701 -> 2,619 (82 dropped, 3.0% of non-null)
  density_weight_lb [2-7]: 341 -> 256 (85 dropped, 24.9% of non-null)
  pocket_depth_in [8-18]: 8,507 -> 3,211 (5,296 dropped, 62.3% of non-null)
  power_rating_w [300-1500]: 6,070 -> 4,096 (1,974 dropped, 32.5% of non-null)
  stage_count_numeric [1-7]: 362 -> 350 (12 dropped, 3.3% of non-null)
  voltage_numeric [110-240]: 2,363 -> 1,928 (435 dropped, 18.4% of non-null)
  thread_count_numeric [150-1000]: 19,346 -> 15,956 (3,390 dropped, 17.5% of non-null)
  weight_numeric [1-50]: 252 -> 235 (17 dropped, 6.7% of non-null)


In [111]:
# Side-by-side: original vs cleaned for one numeric column
col = 'power_rating_w'
if col in df_clean.columns:
    sample = df_clean[df_clean[col].notna()][[col, f'{col}_cleaned']].head(10)
    print(sample)
else:
    print(f'{col} not in df; pick another from VALID_RANGES.')

     power_rating_w  power_rating_w_cleaned
40            130.0                     NaN
619           500.0                   500.0
885             2.0                     NaN
888          1150.0                  1150.0
890           430.0                   430.0
901           430.0                   430.0
907            20.0                     NaN
909           650.0                   650.0
916            60.0                     NaN
917             5.0                     NaN


## 6. Save the final dataframe

Persist `df_clean` (cleaned text + per-source dicts + merged dict + expanded fields + numeric/unit + dimensions + `_cleaned` numerics) to `data/df_features.pkl` so the embedding-analysis notebook can pick it up directly.

In [112]:
from feature_extraction_workflow import save_features

out_path = DATA_DIR / 'df_features.pkl'
save_features(df_clean, out_path)

print(f'Saved {len(df_clean):,} rows x {df_clean.shape[1]} cols -> {out_path}')
print(f'\nIncluded columns include:')
print(f'  product id : asin')
print(f'  text       : ' + ', '.join(c for c in ["title", "description", "feature"] if c in df_clean.columns))
print(f'  cleaned txt: ' + ', '.join(c for c in df_clean.columns if c.endswith("_cleaned") and c.split("_")[0] in ["title", "description", "feature"]))
print(f'  per-source : ' + ', '.join(c for c in df_clean.columns if c.startswith("extracted_features_")))
print(f'  merged     : extracted_features')
print(f'  field cols : ' + str(len([c for c in df_clean.columns if c[:1].isupper()])) + ' columns (Brand, Color, Material, ...)')
print(f'  numerics   : ' + ', '.join(c for c in df_clean.columns if c.endswith("_numeric") or c.endswith("_w") or c.endswith("_lb") or c.endswith("_in")))
print(f'  numerics_cl: ' + ', '.join(c for c in df_clean.columns if c.endswith("_cleaned") and c not in ["title_cleaned", "description_cleaned", "feature_cleaned"]))

Saved 1,134,566 rows x 81 cols -> /Users/lazr/PycharmProjects/RecSystem/data/df_features.pkl

Included columns include:
  product id : asin
  text       : title, description, feature
  cleaned txt: title_cleaned, description_cleaned, feature_cleaned
  per-source : extracted_features_title, extracted_features_description, extracted_features_feature
  merged     : extracted_features
  field cols : 26 columns (Brand, Color, Material, ...)
  numerics   : capacity_numeric, capacity_volume_numeric, piece_count_numeric, thread_count_numeric, weight_numeric, bar_pressure_numeric, capacity_cups_numeric, density_weight_lb, pocket_depth_in, power_rating_w, stage_count_numeric, voltage_numeric
  numerics_cl: bar_pressure_numeric_cleaned, capacity_cups_numeric_cleaned, density_weight_lb_cleaned, pocket_depth_in_cleaned, power_rating_w_cleaned, stage_count_numeric_cleaned, voltage_numeric_cleaned, thread_count_numeric_cleaned, weight_numeric_cleaned
